# Supervised Fine-Tuning (SFT) with NeMo AutoModel

This tutorial walks through **full supervised fine-tuning** of a
pre-trained language model using [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel).

We will fine-tune **Llama-3.2-1B** on the [SQuAD](https://huggingface.co/datasets/rajpurkar/squad)
question-answering dataset, save checkpoints in SafeTensors format,
and run inference on the resulting model.

## Learning Goals

**What is SFT?**

Supervised Fine-Tuning (SFT) takes a pre-trained foundation model and trains
*all* of its weights on labeled instruction-response pairs. Unlike continued
pre-training (DAPT), SFT uses structured data -- typically formatted as
prompt/completion pairs or multi-turn conversations -- and the loss is
**masked so that only the response tokens contribute to the gradient**.
This teaches the model to follow instructions rather than simply predict
the next token in raw text.

**When should you use SFT?**

* You want the model to follow instructions, answer questions, or generate
  structured outputs.
* You have enough labeled data (hundreds to tens of thousands of examples).
* You have sufficient GPU memory to hold the full model plus optimizer
  states. For very large models or limited hardware, consider
  [PEFT/LoRA](peft.ipynb) instead.

**Prerequisites:**

* **NeMo AutoModel** installed (see the
  [AutoModel README](https://github.com/NVIDIA-NeMo/Automodel) for
  installation instructions).
* **Hugging Face token** for gated model access (Llama). Run `huggingface-cli login` or set the `HF_TOKEN` environment variable.

---
## Step 1. Prepare the Data

NeMo AutoModel ships with several built-in dataset helpers:

| Dataset class | Use case |
|---|---|
| `squad.make_squad_dataset` | Extractive QA (SQuAD) |
| `hellaswag.HellaSwag` | Commonsense reasoning |
| `chat_dataset.ChatDataset` | Any data in OpenAI chat-message format |

In this tutorial we use the **SQuAD** dataset. Each example contains a
Wikipedia passage (*context*), a *question*, and one or more reference
*answers*. The dataset helper formats these into prompt-completion pairs
and applies the appropriate loss mask automatically.

In [ ]:
from datasets import load_dataset

ds = load_dataset("rajpurkar/squad", split="train")
print(f"Training samples: {len(ds)}")
print(f"\nExample:")
print(f"  Context: {ds[0]['context'][:200]}...")
print(f"  Question: {ds[0]['question']}")
print(f"  Answer: {ds[0]['answers']['text'][0]}")

### Using Your Own Data with ChatDataset

If you have custom instruction-tuning data, the easiest path is to format
it as a **JSONL file** where each line contains an OpenAI-style
`messages` array. Here is an example of the expected format:

In [ ]:
import json

example = {
    "messages": [
        {"role": "system", "content": "You are a helpful assistant."},
        {"role": "user", "content": "What is machine learning?"},
        {"role": "assistant", "content": "Machine learning is a subset of AI..."}
    ]
}
print(json.dumps(example, indent=2))

To use your own JSONL data, update the `dataset` section of the YAML config:

```yaml
dataset:
  _target_: nemo_automodel.components.datasets.llm.chat_dataset.ChatDataset
  path_or_dataset_id: /path/to/train.jsonl
```

The `ChatDataset` class handles tokenization, chat-template application,
and loss masking (only assistant turns contribute to the loss).

---
## Step 2. Configure SFT

All training parameters live in a single YAML file. Let's inspect the
SFT configuration that ships with this tutorial.

In [ ]:
from pathlib import Path

print(Path("sft_config.yaml").read_text())

### Key Configuration Parameters

| Parameter | Value | Description |
|---|---|---|
| `model._target_` | `from_pretrained` | Load a Hugging Face checkpoint directly |
| `distributed.tp_size` | `2` | Tensor parallelism -- use for 8B+ models or limited per-GPU memory |
| `loss_fn._target_` | `MaskedCrossEntropy` | Cross-entropy loss applied only to response tokens |
| `step_scheduler.max_steps` | `100` | Demo setting -- increase for real training |
| `optimizer.lr` | `5e-6` | Conservative learning rate for full SFT |
| `packed_sequence.packed_sequence_size` | `0` | Disabled -- set > 0 to pack short sequences for throughput |

---
## Step 3. Launch SFT

NeMo AutoModel uses the `automodel` CLI. The `--nproc-per-node` flag
controls how many GPUs to use (must be >= `tp_size`).

In [ ]:
!automodel tutorials/sft-peft/sft_config.yaml

---
## Step 4. Generate from the SFT Checkpoint

After training, checkpoints are saved in SafeTensors format and can be
loaded directly with Hugging Face `transformers`.

In [ ]:
import glob
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

# Find latest checkpoint
ckpt_dirs = sorted(glob.glob("checkpoints/sft/epoch_*_step_*"))
if not ckpt_dirs:
    print("WARNING: No checkpoints found under checkpoints/sft/. Falling back to the base model.")
    latest = "meta-llama/Llama-3.2-1B"
else:
    latest = ckpt_dirs[-1] + "/model/consolidated"
print(f"Loading checkpoint: {latest}")

tokenizer = AutoTokenizer.from_pretrained(latest)
model = AutoModelForCausalLM.from_pretrained(latest, torch_dtype=torch.bfloat16, device_map="auto")

prompt = "Context: The Eiffel Tower is located in Paris, France. Question: Where is the Eiffel Tower?"
inputs = tokenizer(prompt, return_tensors="pt").to(model.device)
with torch.no_grad():
    output = model.generate(**inputs, max_new_tokens=64)
print(tokenizer.decode(output[0], skip_special_tokens=True))

---
## Step 5. Evaluate

For systematic evaluation on standard benchmarks (MMLU, HellaSwag,
IFEval, etc.), see the [Evaluation tutorial](../evaluation/).

Quick sanity checks you can do right now:

* **Perplexity**: compute `exp(loss)` on a held-out split from the
  training logs.
* **Exact Match / F1**: compare generated answers to the SQuAD
  reference answers using the official SQuAD evaluation script.
* **Human spot-check**: generate answers for a handful of questions and
  review them manually.

---
## Next Steps

* **PEFT / LoRA** -- Try [parameter-efficient fine-tuning](peft.ipynb)
  for memory-efficient adaptation on a single GPU.
* **Evaluation** -- Run the [Evaluation tutorial](../evaluation/) to
  benchmark your fine-tuned model.
* **RLHF** -- For reinforcement learning from human feedback (DPO, PPO),
  see [NeMo-RL](https://github.com/NVIDIA/NeMo-RL).

---
## Resources

* [NeMo AutoModel](https://github.com/NVIDIA-NeMo/Automodel) -- the
  training framework used in this tutorial.
* [NeMo Curator](https://github.com/NVIDIA/NeMo-Curator) -- tools for
  large-scale data curation and preprocessing.
* [NeMo-RL](https://github.com/NVIDIA/NeMo-RL) -- RLHF, DPO, and PPO
  alignment pipelines.
* [Hugging Face Transformers](https://huggingface.co/docs/transformers) --
  for checkpoint loading and inference.
* [Llama 3.2 Model Card](https://huggingface.co/meta-llama/Llama-3.2-1B) --
  base model used in this tutorial.